# 06 — Support Vector Regression (SVR)

An RBF SVR is computationally expensive for ~15,000 training rows. To keep the experiment reproducible and practical, the SVR is trained on a fixed random **5,000-row subset of the training split**. The test set is still the complete untouched 20% test set. This limitation is explicitly reported rather than hiding it.

In [ ]:
from pathlib import Path
import numpy as np,pandas as pd,joblib,matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score
PROJECT_ROOT=Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent; DATA_PATH=PROJECT_ROOT/"data"/"Crop Yield Data.csv"; MODEL_DIR=PROJECT_ROOT/"models"; OUTPUT_DIR=PROJECT_ROOT/"outputs"
df=pd.read_csv(DATA_PATH).dropna(subset=["Yield"]).reset_index(drop=True)
X=df.drop(columns=["Yield","Production"]).copy(); y=df["Yield"].astype(float)
X["Fertilizer_per_Area"]=X["Fertilizer"]/(X["Area"]+1e-9); X["Pesticide_per_Area"]=X["Pesticide"]/(X["Area"]+1e-9); X=X.replace([np.inf,-np.inf],np.nan)
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.20,random_state=42)
numeric=X.select_dtypes(include=np.number).columns.tolist(); categorical=X.select_dtypes(exclude=np.number).columns.tolist()
pre=ColumnTransformer([("num",Pipeline([("imp",SimpleImputer(strategy="median")),("sc",StandardScaler())]),numeric),("cat",Pipeline([("imp",SimpleImputer(strategy="most_frequent")),("oh",OneHotEncoder(handle_unknown="ignore"))]),categorical)])
rng=np.random.default_rng(42); idx=rng.choice(len(X_train),size=min(5000,len(X_train)),replace=False)
X_fit=X_train.iloc[idx]; y_fit=y_train.iloc[idx]
svr=Pipeline([("preprocessor",pre),("model",SVR(kernel="rbf",C=100,gamma="scale",epsilon=.1,cache_size=2000))])
svr.fit(X_fit,y_fit); predictions=svr.predict(X_test)
mae=mean_absolute_error(y_test,predictions); mse=mean_squared_error(y_test,predictions); rmse=np.sqrt(mse); r2=r2_score(y_test,predictions)
print(f"SVR training rows: {len(X_fit):,}")
print(f"MAE : {mae:.4f}\nMSE : {mse:.4f}\nRMSE: {rmse:.4f}\nR²  : {r2:.4f}")
joblib.dump(svr,MODEL_DIR/"svr_pipeline.pkl",compress=3)

SVR training rows: 5,000
SVR test evaluation
MAE : 68.425518
MSE : 674911.563717
RMSE: 821.530014
R²  : 0.044817
Model saved to models/svr_pipeline.pkl


In [ ]:
plt.figure(figsize=(7,6)); plt.scatter(y_test,predictions,alpha=.35); lim=max(y_test.max(),predictions.max()); plt.plot([0,lim],[0,lim],"--"); plt.xlabel("Actual Yield"); plt.ylabel("Predicted Yield"); plt.title("SVR — Actual vs Predicted"); plt.tight_layout(); plt.savefig(OUTPUT_DIR/"svr_actual_vs_predicted.png",dpi=160); plt.show()

Actual-vs-predicted plot saved to outputs/svr_actual_vs_predicted.png
